# 03. Check Outputs and Summarize Results

Notebook kiểm tra 105/107 thư mục output sau giai đoạn 1, đọc summary từng run và tạo CSV tổng hợp để viết bài báo.


In [ ]:
from google.colab import drive
drive.mount("/content/drive")

from pathlib import Path
import json
import pandas as pd

PROJECT_DIR = Path("/content/drive/MyDrive/research_ts2img_adaptive_fusion")
OUTPUT_DIR = PROJECT_DIR / "outputs"
if not OUTPUT_DIR.exists():
    raise FileNotFoundError(f"Không tìm thấy {OUTPUT_DIR}")

DATASETS = ["Coffee", "ECG200", "GunPoint", "FordA", "Wafer"]
EXPERIMENTS = ["cnn1d","gaf_lightcnn","mtf_lightcnn","rp_lightcnn","stft_lightcnn","manual_feature_concat","adaptive_fusion_full"]
SEEDS = [42, 43, 44]

runs = sorted([p for p in OUTPUT_DIR.iterdir() if p.is_dir()])
print("Số thư mục output:", len(runs))
print("20 thư mục đầu:")
for p in runs[:20]:
    print("-", p.name)

expected = set()
for d in DATASETS:
    for e in EXPERIMENTS:
        for s in SEEDS:
            expected.add(f"{d}_{e}_seed{s}" if e == "cnn1d" else f"{d}_{e}_img64_seed{s}")

actual = {p.name for p in runs}
missing = sorted(expected - actual)
extra = sorted(actual - expected)

print("\nSố run kỳ vọng:", len(expected))
print("Số thư mục thực tế:", len(actual))
print("Thiếu:", len(missing), missing)
print("Thừa/khác tên:", len(extra), extra[:50])

print("\nFile trong 5 thư mục mẫu:")
for r in runs[:5]:
    print("\n==", r.name, "==")
    for f in sorted(r.iterdir()):
        print("-", f.name)

def flat(d, prefix=""):
    out = {}
    for k, v in d.items():
        key = f"{prefix}_{k}" if prefix else k
        if isinstance(v, dict):
            out.update(flat(v, key))
        else:
            out[key] = v
    return out

def load_summary(run_dir: Path):
    json_files = list(run_dir.glob("*.json")) + list(run_dir.glob("**/*.json"))
    keys = ["summary", "result", "metric", "test", "final"]
    json_files = sorted(json_files, key=lambda p: (not any(k in p.name.lower() for k in keys), len(str(p))))
    for jf in json_files:
        try:
            data = json.loads(jf.read_text(encoding="utf-8"))
            if not isinstance(data, dict):
                continue
            data = flat(data)
            if any(k in data for k in ["test_acc", "test_macro_f1", "dataset", "experiment"]):
                data["_run_dir"] = run_dir.name
                data["_source_file"] = str(jf)
                return data
        except Exception:
            pass
    return None

rows, no_summary = [], []
for r in runs:
    row = load_summary(r)
    if row is None:
        no_summary.append(r.name)
    else:
        rows.append(row)

df = pd.DataFrame(rows)
print("\nSố run đọc được summary:", len(df))
print("Số thư mục chưa có summary:", len(no_summary))
if no_summary:
    print(no_summary[:50])

num_cols = ["seed","test_acc","test_macro_f1","test_precision","test_recall","best_val_f1","best_epoch","params","trainable_params","flops","inference_ms_per_sample","total_train_time_sec","image_size","alpha_GAF","alpha_MTF","alpha_RP","alpha_STFT"]
for c in num_cols:
    if c in df.columns:
        df[c] = pd.to_numeric(df[c], errors="coerce")

display(df.head())
df.to_csv(OUTPUT_DIR / "results_all_runs.csv", index=False)

metrics = ["test_acc","test_macro_f1","test_precision","test_recall","best_val_f1","params","trainable_params","flops","inference_ms_per_sample","total_train_time_sec"]
metrics = [c for c in metrics if c in df.columns]

if "dataset" not in df.columns or "experiment" not in df.columns:
    raise KeyError("Thiếu cột dataset hoặc experiment trong summary JSON.")

summary = df.groupby(["dataset", "experiment"])[metrics].agg(["mean", "std", "count"]).reset_index()
summary.columns = ["_".join(c).strip("_") if isinstance(c, tuple) else c for c in summary.columns]
summary.to_csv(OUTPUT_DIR / "results_mean_std.csv", index=False)
display(summary)

pretty = summary[["dataset","experiment"]].copy()
for m in ["test_acc","test_macro_f1","inference_ms_per_sample","total_train_time_sec"]:
    mc, sc = f"{m}_mean", f"{m}_std"
    if mc in summary.columns and sc in summary.columns:
        pretty[m] = summary.apply(lambda r: f"{r[mc]:.4f} ± {r[sc]:.4f}" if pd.notna(r[sc]) else f"{r[mc]:.4f}", axis=1)
for m in ["params","flops"]:
    mc = f"{m}_mean"
    if mc in summary.columns:
        pretty[m] = summary[mc].round(2)
pretty.to_csv(OUTPUT_DIR / "results_mean_std_pretty.csv", index=False)
display(pretty)

rank_df = summary.sort_values(["dataset", "test_macro_f1_mean"], ascending=[True, False]).copy()
rank_df["rank_by_macro_f1"] = rank_df.groupby("dataset")["test_macro_f1_mean"].rank(method="dense", ascending=False).astype(int)
rank_df.to_csv(OUTPUT_DIR / "results_rank_by_dataset.csv", index=False)
display(rank_df)

best = rank_df.groupby("dataset").head(1).reset_index(drop=True)
best.to_csv(OUTPUT_DIR / "best_by_dataset.csv", index=False)
display(best)

if "test_macro_f1" in df.columns:
    bad = df[df["test_macro_f1"] < 0.40].copy()
    bad.to_csv(OUTPUT_DIR / "low_macro_f1_runs.csv", index=False)
    print("Số run Macro-F1 < 0.40:", len(bad))
    display(bad)

alpha_cols = [c for c in df.columns if c.startswith("alpha_")]
if alpha_cols:
    adf = df[df["experiment"] == "adaptive_fusion_full"].copy()
    alpha_summary = adf.groupby("dataset")[alpha_cols].agg(["mean", "std"]).reset_index()
    alpha_summary.columns = ["_".join(c).strip("_") if isinstance(c, tuple) else c for c in alpha_summary.columns]
    alpha_summary.to_csv(OUTPUT_DIR / "fusion_weights_summary.csv", index=False)
    display(alpha_summary)

    alpha_mean = adf.groupby("dataset")[alpha_cols].mean().reset_index()
    alpha_mean["dominant_fusion_representation"] = alpha_mean[alpha_cols].idxmax(axis=1).str.replace("alpha_", "", regex=False)
    alpha_mean.to_csv(OUTPUT_DIR / "dominant_fusion_representation.csv", index=False)
    display(alpha_mean)

print("\nCSV đã tạo trong:", OUTPUT_DIR)
for name in ["results_all_runs.csv","results_mean_std.csv","results_mean_std_pretty.csv","results_rank_by_dataset.csv","best_by_dataset.csv","low_macro_f1_runs.csv","fusion_weights_summary.csv","dominant_fusion_representation.csv"]:
    print(name, "OK" if (OUTPUT_DIR / name).exists() else "NOT FOUND")
